# DICOM MONOCHROME1 and window (VOI) check

This notebook checks how the 2-D pipeline (`_base_load_2d`, which reads DICOMs with `ITKReader`) handles
MONOCHROME1 images and DICOM window tags, and shows the affected images.

Background:

- ITK (GDCM) flips MONOCHROME1 images while reading, so they come out MONOCHROME2 style (air dark).
  It computes `slope * (M - raw) + intercept`, where `M = 2**BitsStored - 1`, or `2**BitsAllocated - 1` for JPEG 2000.
- The window tags (`WindowCenter` / `WindowWidth`) refer to the **unflipped** values. The old `_apply_voi_lut`
  applied them to the flipped values, which gives wrong contrast when the window is not centered.
  The fixed `_apply_voi_lut` undoes the flip, applies the window, and flips back.
- Some files have no `HighBit` tag. ITK cannot decode them and returns a constant (blank) image.
  The new `_fix_missing_highbit` step re-reads only those files with pydicom, in the same form ITK uses.

Sections:

1. Scan CheXpert-Plus DICOM headers
2. Window fix: old vs fixed vs reference (CheXpert-Plus)
3. JPEG 2000 flip constant (VinDr-CXR)
4. Files with no `HighBit` tag load as blank images (CheXpert-Plus)
5. VinDr-CXR `.dicom` files skip the window step
6. Summary

The **reference** image is built with pydicom only: raw pixels, then rescale, then window, then flip MONOCHROME1,
then the same 0.5 / 99.5 percentile scaling to [-1, 1] that the pipeline uses.

> The outputs of this notebook contain patient images. Clear outputs before committing.

In [ ]:
import glob
import os
import random
import tempfile

import itk
import matplotlib.pyplot as plt
import monai as mn
import numpy as np
import pandas as pd
import pydicom
import torch
from pydicom.pixels import apply_modality_lut, apply_voi_lut

from radharmony.dataset.base import _apply_voi_lut
from radharmony.dataset.transforms import _base_load_2d

CXP_DIR = (
    "/mnt/NAS4/datasets/external/CheXpert_Plus/chexpertplus/DICOM/Uncompressed/train/"
)
VINDR_DIR = (
    "/mnt/NAS4/datasets/external/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/test/"
)

N_PATIENTS = 1500  # CheXpert-Plus patients to sample (about 5,000 files, a few minutes)
N_SHOW = 4  # images to plot per section
SEED = 1

# _apply_voi_lut only runs on paths ending in .dcm, so VinDr files are symlinked here as .dcm
LINK_DIR = tempfile.mkdtemp(prefix="voi_verify_")

### Helpers

`run_pipeline` runs the real 2-D pipeline, swapping only the window step (and optionally turning off the HighBit fix). `_apply_voi_lut_old` is a copy of the
function before the fix.

In [ ]:
_steps = _base_load_2d(["img"])
# [load, channel, slice, transpose, _fix_missing_highbit, window, scale]
RAW_LOAD = _steps[:-3]  # ITK only, before the HighBit fix
LOAD, SCALE = _steps[:-2], _steps[-1]  # LOAD includes the HighBit fix


def _apply_voi_lut_old(x):
    """_apply_voi_lut before the fix: windows the ITK-flipped values directly."""
    path = str(x.meta.get("filename_or_obj", ""))
    if not path.lower().endswith(".dcm"):
        return x
    ds = pydicom.dcmread(path, stop_before_pixels=True)
    arr = apply_voi_lut(x.numpy().squeeze(), ds).astype("float32")
    x[:] = torch.from_numpy(arr).unsqueeze(0)
    return x


def run_pipeline(path, voi_fn=_apply_voi_lut, highbit_fix=True):
    """Pipeline output in [-1, 1] with the given window step, with or without the HighBit fix."""
    load = LOAD if highbit_fix else RAW_LOAD
    t = mn.transforms.Compose(
        load + [mn.transforms.Lambdad(keys=["img"], func=voi_fn), SCALE]
    )
    return t({"img": path})["img"].numpy().squeeze()


def load_itk(path):
    """Pixels exactly as ITK reads them (no HighBit fix, no window, no scaling)."""
    return mn.transforms.Compose(RAW_LOAD)({"img": path})["img"].numpy().squeeze()


def reference(path):
    """Correct image from pydicom: rescale, window, flip MONOCHROME1, percentile-scale to [-1, 1]."""
    ds = pydicom.dcmread(path)
    ref = apply_voi_lut(
        apply_modality_lut(ds.pixel_array.astype(float), ds), ds
    ).astype(float)
    if ds.PhotometricInterpretation == "MONOCHROME1":
        ref = ref.max() - ref
    lo, hi = np.percentile(ref, [0.5, 99.5])
    return np.clip((ref - lo) / (hi - lo) * 2 - 1, -1, 1)


def as_dcm(path):
    """Symlink a .dicom file as .dcm so _apply_voi_lut does not skip it."""
    link = os.path.join(LINK_DIR, os.path.basename(path).rsplit(".", 1)[0] + ".dcm")
    if not os.path.lexists(link):
        os.symlink(path, link)
    return link


def show_row(axes, images, titles):
    for ax, img, t in zip(axes, images, titles):
        ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
        ax.set_title(t, fontsize=9)
        ax.axis("off")


def header_row(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True)
    wc = ds.get("WindowCenter")
    wc = (
        float(wc[0])
        if isinstance(wc, pydicom.multival.MultiValue)
        else (float(wc) if wc is not None else None)
    )
    ww = ds.get("WindowWidth")
    ww = (
        float(ww[0])
        if isinstance(ww, pydicom.multival.MultiValue)
        else (float(ww) if ww is not None else None)
    )
    bits = int(ds.BitsStored)
    return dict(
        path=path,
        pi=ds.PhotometricInterpretation,
        bits_stored=bits,
        bits_alloc=int(ds.BitsAllocated),
        has_highbit="HighBit" in ds,
        has_window=wc is not None,
        wc=wc,
        ww=ww,
        slope=float(ds.get("RescaleSlope", 1) or 1),
        intercept=float(ds.get("RescaleIntercept", 0) or 0),
        j2k=str(ds.file_meta.TransferSyntaxUID) == "1.2.840.10008.1.2.4.90",
        off_center=wc is not None and abs(wc - 2**bits / 2) > 1,
    )

---
## 1. Scan CheXpert-Plus DICOM headers

Reads headers only (no pixels) for a random sample of patients. Files that are not valid DICOM (leftover
`.azDownload-*` partial downloads) are counted separately.

In [ ]:
random.seed(SEED)
patients = random.sample(sorted(os.listdir(CXP_DIR)), N_PATIENTS)
rows, invalid = [], []
for p in patients:
    for f in glob.glob(os.path.join(CXP_DIR, p, "*", "*.dcm"), include_hidden=True):
        try:
            rows.append(header_row(f))
        except pydicom.errors.InvalidDicomError:
            invalid.append(f)
cxp = pd.DataFrame(rows)
print(f"{len(cxp)} valid files, {len(invalid)} not valid DICOM")
if invalid:
    print("example invalid:", invalid[0])
cxp.groupby(["pi", "has_window", "off_center", "has_highbit"]).size().rename(
    "files"
).to_frame()

---
## 2. Window fix: old vs fixed vs reference

For every MONOCHROME1 file with window tags, compare the old and fixed pipeline against the reference.
`diff` is the mean absolute difference on the [-1, 1] scale.

- **old** = the code before both fixes (raw ITK pixels + old window step).
- **fixed** = the current pipeline (`_fix_missing_highbit` + fixed `_apply_voi_lut`).
- Files with no `HighBit` are included: under the old code they are blank (Section 4), so they need both fixes.
- A few MONOCHROME2 files with `HighBit` are included as a control: old and fixed should be the same there.

In [ ]:
m1 = cxp[(cxp.pi == "MONOCHROME1") & cxp.has_window]
m2 = cxp[(cxp.pi == "MONOCHROME2") & cxp.has_window & cxp.has_highbit].head(10)

res = []
for _, r in pd.concat([m1, m2]).iterrows():
    ref = reference(r.path)
    old = run_pipeline(r.path, _apply_voi_lut_old, highbit_fix=False)
    new = run_pipeline(r.path, _apply_voi_lut)
    res.append(
        dict(
            path=r.path,
            pi=r.pi,
            has_highbit=r.has_highbit,
            off_center=r.off_center,
            wc=r.wc,
            ww=r.ww,
            slope=r.slope,
            intercept=r.intercept,
            diff_old=np.abs(old - ref).mean(),
            diff_new=np.abs(new - ref).mean(),
        )
    )
res = pd.DataFrame(res)
res["old_ok"] = res.diff_old < 0.01
res["new_ok"] = res.diff_new < 0.01
res.groupby(["pi", "has_highbit", "off_center"])[["old_ok", "new_ok"]].agg(
    ["sum", "count"]
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
r1 = res[res.pi == "MONOCHROME1"]
ax.hist([r1.diff_old, r1.diff_new], bins=30, label=["old", "fixed"])
ax.set_xlabel("mean abs diff vs reference ([-1, 1] scale)")
ax.set_ylabel("MONOCHROME1 files")
ax.set_yscale("log")
ax.legend()
plt.show()

### Worst MONOCHROME1 files under the old code (window problem only)

Files with `HighBit`, so the error comes only from the window step.
Columns: reference, old pipeline, fixed pipeline, and the old error map `|old - reference|`.

In [ ]:
def plot_old_fixed(rows):
    fig, axes = plt.subplots(len(rows), 4, figsize=(13, 3.4 * len(rows)), squeeze=False)
    for row, (_, r) in zip(axes, rows.iterrows()):
        ref = reference(r.path)
        old = run_pipeline(r.path, _apply_voi_lut_old, highbit_fix=False)
        new = run_pipeline(r.path, _apply_voi_lut)
        show_row(
            row[:3],
            [ref, old, new],
            [
                "reference",
                f"old (diff {r.diff_old:.3f})",
                f"fixed (diff {r.diff_new:.3f})",
            ],
        )
        row[3].imshow(np.abs(old - ref), cmap="magma", vmin=0, vmax=2)
        row[3].set_title("|old - reference|", fontsize=9)
        row[3].axis("off")
        row[0].text(
            0,
            -0.08,
            f"WC {r.wc:g} / WW {r.ww:g}, slope {r.slope:g}, int {r.intercept:g}\n"
            + r.path.replace(CXP_DIR, ""),
            transform=row[0].transAxes,
            fontsize=7,
            va="top",
        )
    plt.tight_layout()
    plt.show()


r1 = res[res.pi == "MONOCHROME1"]
plot_old_fixed(r1[r1.has_highbit].sort_values("diff_old", ascending=False).head(N_SHOW))

### MONOCHROME1 files with window tags and no `HighBit` (both fixes)

Under the old code these are blank. The fixed pipeline re-reads them with pydicom, then applies the window fix.

In [ ]:
nohb_win = r1[~r1.has_highbit].head(N_SHOW)
print(
    f"{(~r1.has_highbit).sum()} such files; fixed matches reference: {nohb_win.new_ok.sum()} / {len(nohb_win)} shown, "
    f"{r1[~r1.has_highbit].new_ok.sum()} / {(~r1.has_highbit).sum()} total"
)
if len(nohb_win):
    plot_old_fixed(nohb_win)

---
## 3. JPEG 2000 flip constant (VinDr-CXR)

For JPEG 2000 files GDCM flips around `2**BitsAllocated - 1` (65535) instead of `2**BitsStored - 1` (4095).
pydicom has no JPEG 2000 decoder in this environment, so there is no reference image. Instead, check that
undoing the flip with 65535 puts the values back in `0 .. 2**BitsStored - 1`, and compare old vs fixed visually.
The old code windows ITK values of 61440..65535 with a window like WC 2047 / WW 4095, so every pixel saturates
and the image comes out blank (the "Divide by zero" warning below is the percentile scaling hitting a flat image).
In the real pipeline this does not happen yet, because VinDr files end in `.dicom` (Section 5).

In [ ]:
vindr_files = sorted(glob.glob(os.path.join(VINDR_DIR, "*.dicom")))
vindr = pd.DataFrame([header_row(f) for f in vindr_files])
print(f"{len(vindr)} VinDr test files")
vindr[vindr.pi == "MONOCHROME1"].groupby(["j2k", "has_window"]).size().rename(
    "files"
).to_frame()

In [ ]:
j2k = vindr[(vindr.pi == "MONOCHROME1") & vindr.j2k & vindr.has_window].head(20)
checks = []
for _, r in j2k.iterrows():
    itk_px = load_itk(as_dcm(r.path))
    m_alloc = 2**r.bits_alloc - 1
    unflip = (r.slope * m_alloc + 2 * r.intercept) - itk_px
    stored_max = r.slope * (2**r.bits_stored - 1) + r.intercept
    checks.append(
        dict(
            file=os.path.basename(r.path),
            itk_min=itk_px.min(),
            itk_max=itk_px.max(),
            unflip_min=unflip.min(),
            unflip_max=unflip.max(),
            fits=unflip.min() >= r.intercept and unflip.max() <= stored_max,
        )
    )
checks = pd.DataFrame(checks)
print("all un-flipped ranges fit:", checks.fits.all())
checks.head()

In [ ]:
fig, axes = plt.subplots(
    min(N_SHOW, len(j2k)), 2, figsize=(7, 3.4 * min(N_SHOW, len(j2k))), squeeze=False
)
for row, (_, r) in zip(axes, j2k.head(N_SHOW).iterrows()):
    link = as_dcm(r.path)
    show_row(
        row,
        [run_pipeline(link, _apply_voi_lut_old), run_pipeline(link, _apply_voi_lut)],
        [f"old  (WC {r.wc:g} / WW {r.ww:g})", "fixed"],
    )
plt.tight_layout()
plt.show()

---
## 4. Files with no `HighBit` tag: blank in ITK, fixed by `_fix_missing_highbit`

ITK (GDCM) cannot read these correctly: every pixel comes out the same value. pydicom reads them fine
(it assumes `HighBit = BitsStored - 1`). `_fix_missing_highbit` re-reads only these files with pydicom.
Check: how many load blank in raw ITK, and whether the fixed pipeline matches the reference.

In [ ]:
nohb = cxp[~cxp.has_highbit].copy()
nohb["itk_flat"] = [bool(np.ptp(load_itk(p)) == 0) for p in nohb.path]
nohb["diff_fixed"] = [np.abs(run_pipeline(p) - reference(p)).mean() for p in nohb.path]
nohb["fixed_ok"] = nohb.diff_fixed < 0.01
print(
    f"{len(nohb)} of {len(cxp)} sampled files have no HighBit; "
    f"{nohb.itk_flat.sum()} of them load flat in raw ITK "
    f"({100 * nohb.itk_flat.sum() / len(cxp):.1f}% of all sampled files)"
)
print(f"fixed pipeline matches reference: {nohb.fixed_ok.sum()} / {len(nohb)}")
nohb.groupby(["pi", "itk_flat", "fixed_ok"]).size().rename("files").to_frame()

In [ ]:
flat = nohb[nohb.itk_flat].head(N_SHOW)
if len(flat):
    fig, axes = plt.subplots(len(flat), 3, figsize=(10, 3.4 * len(flat)), squeeze=False)
    for row, (_, r) in zip(axes, flat.iterrows()):
        itk_px = load_itk(r.path)
        show_row(
            row,
            [
                run_pipeline(r.path, highbit_fix=False),
                run_pipeline(r.path),
                reference(r.path),
            ],
            [
                f"before fix (ITK values all {itk_px.min():g})",
                f"fixed (diff {r.diff_fixed:.3f})",
                "reference (pydicom)",
            ],
        )
        row[0].text(
            0,
            -0.08,
            r.path.replace(CXP_DIR, ""),
            transform=row[0].transAxes,
            fontsize=7,
            va="top",
        )
    plt.tight_layout()
    plt.show()

---
## 5. VinDr-CXR `.dicom` files skip the window step

`_apply_voi_lut` returns early unless the path ends in `.dcm`, so VinDr images are never windowed.
They are **not** inverted (ITK already flipped MONOCHROME1); they just ignore the window tags.
Left: current pipeline on the real `.dicom` path. Right: same file through the fixed window step.

In [ ]:
unc = vindr[(vindr.pi == "MONOCHROME1") & ~vindr.j2k & vindr.has_window]
print(
    f"{vindr.has_window.sum()} of {len(vindr)} VinDr test files have window tags and are never windowed"
)
fig, axes = plt.subplots(
    min(N_SHOW, len(unc)), 3, figsize=(10, 3.4 * min(N_SHOW, len(unc))), squeeze=False
)
for row, (_, r) in zip(axes, unc.head(N_SHOW).iterrows()):
    cur = run_pipeline(r.path)  # .dicom: window step skipped
    win = run_pipeline(as_dcm(r.path))  # .dcm symlink: window applied
    ref = reference(r.path)
    show_row(
        row,
        [cur, win, ref],
        [
            f"current, no window (diff {np.abs(cur - ref).mean():.3f})",
            f"windowed WC {r.wc:g} / WW {r.ww:g} (diff {np.abs(win - ref).mean():.3f})",
            "reference",
        ],
    )
plt.tight_layout()
plt.show()

---
## 6. Summary

In [ ]:
r1 = res[res.pi == "MONOCHROME1"]
summary = pd.DataFrame(
    [
        ("CheXpert-Plus files sampled", len(cxp)),
        ("  not valid DICOM (.azDownload leftovers)", len(invalid)),
        ("  MONOCHROME1 with window tags", len(r1)),
        ("    wrong under old code (diff >= 0.01)", int((~r1.old_ok).sum())),
        ("    wrong under fixed code", int((~r1.new_ok).sum())),
        (
            "  MONOCHROME2 controls changed by fix",
            int(
                (
                    res[res.pi == "MONOCHROME2"].diff_old.round(6)
                    != res[res.pi == "MONOCHROME2"].diff_new.round(6)
                ).sum()
            ),
        ),
        ("  no HighBit", len(nohb)),
        ("    load flat in raw ITK", int(nohb.itk_flat.sum())),
        ("    wrong under fixed code", int((~nohb.fixed_ok).sum())),
        (
            "VinDr test JPEG 2000 MONOCHROME1 windowed, un-flip fits range",
            f"{int(checks.fits.sum())} / {len(checks)}",
        ),
        (
            "VinDr test files with window tags that are never windowed",
            int(vindr.has_window.sum()),
        ),
    ],
    columns=["check", "value"],
)
summary